# Picomoteur 8311 sur 8743-CL : preuve de concept

Ce notebook charge le contrôleur, pose une référence quelque part, note deux positions A et B, puis va de A à B.

**Avant de lancer**
- Application Picomotor fermée. Contrôleur 8743-CL allumé et branché en USB, 8311 sur l'axe 1.
- 8311 libre, sans rien pousser, avec au moins 0,3 mm de course de chaque côté.
- Lancer le notebook depuis le dossier qui contient `newport_picomotors.py`.

**Sûreté**
- Le bouton Stop (carré) du notebook arrête le moteur : le code envoie `ST` à l'interruption.
- Dernière cellule : `pico.close()` libère l'USB. Tant que le kernel garde la connexion, aucun script ne peut s'y connecter.
- `TP` vit dans le contrôleur : A et B restent valables tant qu'il reste allumé, même si le kernel redémarre.

In [1]:
# Cellule 1 : charger le moteur
import os, sys
sys.path.insert(0, os.getcwd())

from qcodes.instrument import Instrument
from newport_picomotors import connect_8743

Instrument.close_all()                 # permet de relancer cette cellule sans erreur de nom
pico = connect_8743("pico", allow_motion=True, positions_file="positions_8743.json")
ax = pico.axis1                        # le 8311
print(pico.get_idn())
ax.where()

Connected to: New_Focus 8743-CL (serial:13097, firmware:4.06) in 1.14s
{'vendor': 'New_Focus', 'model': '8743-CL', 'firmware': '4.06', 'serial': '13097'}


{'counts': 1000000,
 'um_depuis_reference': 0.0,
 'reference_valide': True,
 'en_mouvement': False,
 'boucle_fermee': 0,
 'vitesse_VA': 2000,
 'positions_proches': []}

## Référence

Le 8311 n'a pas de référence propre : `TP` compte depuis la mise sous tension. `set_reference()` fixe l'origine à la position actuelle, sans mouvement. On le fait une fois par démarrage du contrôleur.

In [2]:
if ax.reference_valid:
    print("Référence déjà posée.")
else:
    ax.set_reference()
    print("Référence posée à la position actuelle.")
ax.where()

Référence déjà posée.


{'counts': 1000000,
 'um_depuis_reference': 0.0,
 'reference_valide': True,
 'en_mouvement': False,
 'boucle_fermee': 0,
 'vitesse_VA': 2000,
 'positions_proches': []}

## Noter A, avancer, noter B

A = position actuelle. B = A + 100 µm (2016 comptes, 1 compte = 49,6 nm). `speed` vaut `"lent"` (VA 50), `"normal"` (500) ou `"rapide"` (1000).

In [3]:
ax.mark("A")
r = ax.forward(100, speed="normal", unit="um")
print(r)                               # ecart_comptes : différence entre le déplacement lu et demandé
ax.mark("B")
ax.positions()

{'demande_comptes': 2016, 'TP_avant': 1000000, 'TP_apres': 1002016, 'ecart_comptes': 0, 'duree_s': 6.4, 'VA': 500}


{'A': {'counts': 1000000,
  'um_depuis_reference': 0.0,
  'date': '2026-10-15T17:19:43'},
 'B': {'counts': 1002016,
  'um_depuis_reference': 99.994,
  'date': '2026-10-15T17:19:50'}}

## Aller-retour de A à B

`goto` va à une position mémorisée à `VA` 500. Il arrive toujours par le côté + (recul de 500 comptes si besoin) et active la boucle fermée pendant le déplacement, puis la coupe. Compter environ une minute.

In [4]:
resultats = []
for cycle in range(3):
    for nom in ("B", "A"):
        r = ax.goto(nom)
        resultats.append(r)
        print(f"cycle {cycle + 1}  {nom} : TP {r['TP_final']}, écart {r['ecart_comptes']:+d} comptes, {r['duree_s']} s")
pire = max(abs(r["ecart_comptes"]) for r in resultats)
print(f"Écart maximal sur {len(resultats)} arrivées : {pire} comptes ({pire * 0.0496:.2f} µm)")

cycle 1  B : TP 1002015, écart -1 comptes, 6.9 s
cycle 1  A : TP 1000000, écart +0 comptes, 13.3 s
cycle 2  B : TP 1002016, écart +0 comptes, 8.3 s
cycle 2  A : TP 1000000, écart +0 comptes, 13.3 s
cycle 3  B : TP 1002015, écart -1 comptes, 8.3 s
cycle 3  A : TP 1000000, écart +0 comptes, 13.3 s
Écart maximal sur 6 arrivées : 1 comptes (0.05 µm)


## Déplacements à la main

`nudge` fait un incrément de 20 comptes (environ 1 µm). `forward` et `backward` prennent une distance en `counts` ou en `um`, et une vitesse.

In [5]:
print(ax.nudge(1))                          # +1 incrément
print(ax.nudge(-1, steps=5))                # -5 incréments
print(ax.forward(20, speed="lent", unit="um"))
print(ax.backward(20, speed="lent", unit="um"))
ax.goto("A")
ax.where()

{'demande_comptes': 20, 'TP_avant': 1000000, 'TP_apres': 1000020, 'ecart_comptes': 0, 'duree_s': 0.9, 'VA': 500}
{'demande_comptes': -100, 'TP_avant': 1000020, 'TP_apres': 999920, 'ecart_comptes': 0, 'duree_s': 1.2, 'VA': 500}
{'demande_comptes': 403, 'TP_avant': 999920, 'TP_apres': 1000323, 'ecart_comptes': 0, 'duree_s': 11.5, 'VA': 50}
{'demande_comptes': -403, 'TP_avant': 1000323, 'TP_apres': 999920, 'ecart_comptes': 0, 'duree_s': 13.9, 'VA': 50}


{'counts': 1000000,
 'um_depuis_reference': 0.0,
 'reference_valide': True,
 'en_mouvement': False,
 'boucle_fermee': 0,
 'vitesse_VA': 500,
 'positions_proches': ['A']}

## Fin

Libère l'USB pour les autres scripts. Pour ré-ouvrir, relancer la cellule 1.

In [ ]:
pico.close()